# Fixed client-side detection, estimation and PSO-like control

Run this notebook on the client computer. Run server.ipynb on each JetBot first.

In [ ]:
import cv2
import numpy as np
import requests
import time
import random
from dataclasses import dataclass, field
from urllib.parse import urlencode
from PIL import Image as PILImage
from io import BytesIO
from IPython.display import display, clear_output


In [ ]:
# ----------------------------
# Network / motor functions
# ----------------------------

def clamp(value, low, high):
    return max(low, min(high, value))


def get_camera_frame(jetbot_ip, timeout=1.0):
    """Read one camera frame from a JetBot server and return it as BGR OpenCV image."""
    url = f"http://{jetbot_ip}:8080/camera"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()

    pil_image = PILImage.open(BytesIO(response.content)).convert("RGB")
    frame_rgb = np.array(pil_image)
    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    return frame_bgr


def set_motors(jetbot_ip, left_speed, right_speed, timeout=0.5):
    """Set left and right motor speed. Speeds should normally be between -1 and 1."""
    left_speed = clamp(float(left_speed), -0.6, 0.6)
    right_speed = clamp(float(right_speed), -0.6, 0.6)

    params = {"left": left_speed, "right": right_speed}
    url = f"http://{jetbot_ip}:8080/set_motors?{urlencode(params)}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()


def stop_robot(jetbot_ip, timeout=0.5):
    url = f"http://{jetbot_ip}:8080/stop"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()


def stop_all(jetbot_ips):
    for ip in jetbot_ips:
        try:
            stop_robot(ip)
        except Exception as e:
            print(f"Could not stop {ip}: {e}")


In [ ]:
# ----------------------------
# Detection + distance estimation
# ----------------------------

# Calibration values.
# Put the red box at known_distance_cm from the camera and measure its width in pixels.
known_distance_cm = 30.0
real_target_width_cm = 5.0
measured_box_width_px = 80.0

focal_length_px = (measured_box_width_px * known_distance_cm) / real_target_width_cm
print("Focal length px:", focal_length_px)


def estimate_distance(real_width_cm, focal_length_px, box_width_px):
    if box_width_px <= 0:
        return None
    return (real_width_cm * focal_length_px) / box_width_px


def get_color_masks(hsv):
    masks = {}

    # Red can wrap around hue=0, so two ranges are safer.
    lower_red_1 = np.array([0, 100, 80])
    upper_red_1 = np.array([15, 255, 255])

    lower_red_2 = np.array([165, 100, 80])
    upper_red_2 = np.array([179, 255, 255])

    red_mask_1 = cv2.inRange(hsv, lower_red_1, upper_red_1)
    red_mask_2 = cv2.inRange(hsv, lower_red_2, upper_red_2)

    masks["red"] = cv2.bitwise_or(red_mask_1, red_mask_2)

    return masks


def detect_target(frame_bgr, min_area=500):
    """Detect the largest red target.

    Returns:
        annotated_frame, detection

    detection is None if no target is found.
    Otherwise:
        {
            color, x, y, w, h, cx, cy, area,
            distance_cm, direction_error, score
        }

    direction_error:
        -1.0 means far left, 0 centered, +1.0 far right.
    """

    frame = frame_bgr.copy()
    height, width = frame.shape[:2]

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    masks = get_color_masks(hsv)

    best_detection = None

    for color_name, mask in masks.items():
        small_kernel = np.ones((3, 3), np.uint8)
        big_kernel = np.ones((9, 9), np.uint8)

        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, small_kernel, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, big_kernel, iterations=2)
        mask = cv2.dilate(mask, big_kernel, iterations=1)

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        valid_contours = [
            c for c in contours
            if cv2.contourArea(c) > min_area
        ]

        if not valid_contours:
            continue

        contour = max(valid_contours, key=cv2.contourArea)
        area = float(cv2.contourArea(contour))
        x, y, w, h = cv2.boundingRect(contour)

        cx = x + w / 2.0
        cy = y + h / 2.0

        distance_cm = estimate_distance(
            real_width_cm=real_target_width_cm,
            focal_length_px=focal_length_px,
            box_width_px=w
        )

        # -1 left, 0 center, +1 right
        direction_error = (cx - (width / 2.0)) / (width / 2.0)

        # Score should become larger when the target is larger/closer and more centered.
        centered_bonus = 1.0 - min(abs(direction_error), 1.0)
        score = area * (0.5 + 0.5 * centered_bonus)

        detection = {
            "color": color_name,
            "x": int(x),
            "y": int(y),
            "w": int(w),
            "h": int(h),
            "cx": float(cx),
            "cy": float(cy),
            "area": area,
            "distance_cm": distance_cm,
            "direction_error": float(direction_error),
            "score": float(score),
        }

        if best_detection is None or detection["score"] > best_detection["score"]:
            best_detection = detection

    if best_detection is not None:
        x = best_detection["x"]
        y = best_detection["y"]
        w = best_detection["w"]
        h = best_detection["h"]
        distance_cm = best_detection["distance_cm"]
        direction_error = best_detection["direction_error"]

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.circle(frame, (int(best_detection["cx"]), int(best_detection["cy"])), 4, (0, 255, 255), -1)
        cv2.line(frame, (width // 2, 0), (width // 2, height), (255, 255, 255), 1)

        label = f"red {distance_cm:.1f} cm err={direction_error:.2f}"
        cv2.putText(
            frame,
            label,
            (x, max(20, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 255),
            2
        )

    return frame, best_detection


In [ ]:
# ----------------------------
# PSO-like robot state and controller
# ----------------------------

@dataclass
class RobotState:
    ip: str
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2))  # [forward, turn]
    personal_best_score: float = -1.0
    personal_best_direction: float = 0.0
    last_seen_time: float = 0.0
    last_detection: dict = None


@dataclass
class SwarmState:
    robots: dict
    global_best_score: float = -1.0
    global_best_direction: float = 0.0
    global_best_robot_ip: str = None
    global_best_time: float = 0.0


def update_bests(swarm, robot_state, detection):
    now = time.time()

    if detection is None:
        return

    score = detection["score"]
    direction = detection["direction_error"]

    robot_state.last_seen_time = now
    robot_state.last_detection = detection

    if score > robot_state.personal_best_score:
        robot_state.personal_best_score = score
        robot_state.personal_best_direction = direction

    if score > swarm.global_best_score:
        swarm.global_best_score = score
        swarm.global_best_direction = direction
        swarm.global_best_robot_ip = robot_state.ip
        swarm.global_best_time = now


def pso_visual_controller(robot_state, swarm, detection):
    """Return left_speed, right_speed for a JetBot.

    This is PSO-like because we do not know real x/y coordinates.
    Instead, the velocity vector is [forward_speed, turn_speed].

    The controller uses:
    - current visual target direction
    - the robot's personal best visual direction
    - the swarm's global best visual direction
    - random exploration when target is not visible
    """

    # Hyperparameters
    inertia = 0.45
    cognitive = 0.25
    social = 0.20
    random_gain = 0.20

    base_forward = 0.16
    max_forward = 0.28
    max_turn = 0.25

    target_stop_distance_cm = 18.0
    target_slow_distance_cm = 35.0

    r1 = random.random()
    r2 = random.random()

    old_forward, old_turn = robot_state.velocity

    if detection is not None:
        direction_error = detection["direction_error"]
        distance_cm = detection["distance_cm"]

        # Turn toward the target.
        # Positive direction_error means target is to the right,
        # so turn right by increasing left motor and decreasing right motor.
        target_turn = direction_error

        # Forward speed depends on distance.
        if distance_cm is not None and distance_cm < target_stop_distance_cm:
            target_forward = 0.0
        elif distance_cm is not None and distance_cm < target_slow_distance_cm:
            target_forward = 0.10
        else:
            target_forward = base_forward

        # PSO-like velocity update.
        cognitive_pull = cognitive * r1 * (robot_state.personal_best_direction - old_turn)
        social_pull = social * r2 * (swarm.global_best_direction - old_turn)

        new_turn = (
            inertia * old_turn
            + 0.75 * target_turn
            + cognitive_pull
            + social_pull
        )

        new_forward = (
            inertia * old_forward
            + 0.55 * target_forward
        )

    else:
        # No target visible: search.
        # Robots rotate in slightly different random directions.
        random_turn = random.uniform(-1.0, 1.0)

        # If another robot recently saw the target, bias the search using global best.
        recent_global = (time.time() - swarm.global_best_time) < 3.0

        if recent_global:
            search_turn = 0.6 * swarm.global_best_direction + 0.4 * random_turn
        else:
            search_turn = random_turn

        new_turn = inertia * old_turn + random_gain * search_turn
        new_forward = 0.05

    new_forward = clamp(new_forward, 0.0, max_forward)
    new_turn = clamp(new_turn, -max_turn, max_turn)

    robot_state.velocity = np.array([new_forward, new_turn], dtype=float)

    # Differential drive mixing.
    left_speed = new_forward + new_turn
    right_speed = new_forward - new_turn

    left_speed = clamp(left_speed, -0.35, 0.35)
    right_speed = clamp(right_speed, -0.35, 0.35)

    return left_speed, right_speed


In [ ]:
# ----------------------------
# Run PSO control loop
# ----------------------------

jetbot_ips = [
    "172.20.10.2",
    # "172.20.10.3",
    # "172.20.10.4",
]

robots = {ip: RobotState(ip=ip) for ip in jetbot_ips}
swarm = SwarmState(robots=robots)


def run_swarm_pso(duration_seconds=60, loop_delay=0.10, show_camera=True):
    start_time = time.time()

    try:
        while time.time() - start_time < duration_seconds:
            annotated_frames = []

            for ip, robot_state in robots.items():
                try:
                    frame = get_camera_frame(ip)
                    annotated, detection = detect_target(frame)

                    update_bests(swarm, robot_state, detection)

                    left_speed, right_speed = pso_visual_controller(
                        robot_state=robot_state,
                        swarm=swarm,
                        detection=detection
                    )

                    set_motors(ip, left_speed, right_speed)

                    status = {
                        "ip": ip,
                        "seen": detection is not None,
                        "left": round(left_speed, 3),
                        "right": round(right_speed, 3),
                        "distance": None if detection is None else round(detection["distance_cm"], 1),
                        "direction": None if detection is None else round(detection["direction_error"], 2),
                        "pbest": round(robot_state.personal_best_score, 1),
                    }

                    cv2.putText(
                        annotated,
                        str(status),
                        (10, annotated.shape[0] - 15),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.45,
                        (255, 255, 255),
                        1
                    )

                    annotated_frames.append(annotated)

                    print(status)

                except Exception as e:
                    print(f"{ip}: error -> {e}")
                    try:
                        stop_robot(ip)
                    except Exception:
                        pass

            if show_camera and annotated_frames:
                # Show first robot camera. For several robots, stack horizontally if same size.
                if len(annotated_frames) == 1:
                    show_frame = annotated_frames[0]
                else:
                    min_h = min(f.shape[0] for f in annotated_frames)
                    resized = [
                        cv2.resize(f, (int(f.shape[1] * min_h / f.shape[0]), min_h))
                        for f in annotated_frames
                    ]
                    show_frame = np.hstack(resized)

                show_rgb = cv2.cvtColor(show_frame, cv2.COLOR_BGR2RGB)
                clear_output(wait=True)
                display(PILImage.fromarray(show_rgb))

                print("Global best robot:", swarm.global_best_robot_ip)
                print("Global best score:", round(swarm.global_best_score, 1))
                print("Global best direction:", round(swarm.global_best_direction, 2))

            time.sleep(loop_delay)

    except KeyboardInterrupt:
        print("Stopped manually.")

    finally:
        stop_all(jetbot_ips)
        print("All robots stopped.")


In [ ]:
# Start the PSO-like swarm controller.
# First test with ONE JetBot and low duration.
run_swarm_pso(duration_seconds=30, loop_delay=0.10, show_camera=True)
